# 04 · Is this a verified purchase? (binary classification)

**Use case:** trust & safety wants to flag reviews that read like unverified ones — a binary score per review that can rank the whole table.

**Model / lane:** relational GNN — default GraphSAGE (*Baseline*), text embeddings

**Sub-tasks**
1. Define a **binary** task on `review.verified`
2. Train the relational GNN
3. AUROC, PR-AUC (vs the class-prior baseline), precision / recall / F1
4. Rank all reviews with `predict.top` and score a batch of entity ids

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
p = get_or_create_project(ls, "verified", kind="data_science")
before = credits_used(ls)

reusing project 28106708-9ec0-48e4-b292-d600b9bc2ed8 (amazon-reviews-verified, status=ready)


project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']


In [3]:
model = train_or_reuse(p, "Predict whether a review is a verified purchase, from the review text, the summary, the rating and the product",
                       task_type="supervised", subtask_type="binary_classification", enable_text_embedding=True)
metrics = model["metrics"]
show(metrics, keys=("auroc", "pr_auc", "pr_auc_baseline", "acc", "precision", "recall", "f1"))

reusing model 61b6e350-ea4c-4219-a694-00835a452fd5 (Baseline, trained 2026-09-15T09:02:43.001054+00:00)
  f1                     0.7783
  acc                    0.667
  auroc                  0.6281
  pr_auc                 0.7831
  recall                 0.8453
  precision              0.7212
  pr_auc_baseline        0.6915


In [4]:
print(f"AUROC {metrics.get('auroc'):.3f} (random = 0.500) · PR-AUC {metrics.get('pr_auc'):.3f} vs class prior {metrics.get('pr_auc_baseline', float('nan')):.3f} · F1 {metrics.get('f1'):.3f}")

AUROC 0.628 (random = 0.500) · PR-AUC 0.783 vs class prior 0.692 · F1 0.778


## Rank every review

`predict.top` returns the entities ordered by the positive-class probability — the reviews most likely to be verified (or, with `order="asc"`, least likely).

In [5]:
top = ls.predict.top(model["model_id"], n=10)
print("ranked by:", top.get("ranked_by"), "·", top.get("total_ranked"), "ranked")
for r in top["ranking"][:5]:
    print(f"  #{r['rank']} entity {r['entity_id']} · p={r.get('score')} · percentile {r.get('percentile')}")
least = ls.predict.top(model["model_id"], n=3, order="asc")
print("least likely verified:", [r["entity_id"] for r in least["ranking"]])

ranked by: score · 2000 ranked
  #1 entity 3997 · p=0.9215402603149414 · percentile 100
  #2 entity 723 · p=0.9213626384735107 · percentile 100
  #3 entity 628 · p=0.919954240322113 · percentile 100
  #4 entity 8786 · p=0.9194719195365906 · percentile 100
  #5 entity 2517 · p=0.9187539219856262 · percentile 100


least likely verified: [8105, 6857, 1843]


In [6]:
ids = [r["entity_id"] for r in top["ranking"][:5]]
batch = ls.predict.predict_batch(model_id=model["model_id"], entity_ids=ids)
print(batch["succeeded"], "of", batch["total"], "scored")
for eid, pr in zip(ids, batch["predictions"]):
    print(f"  {eid}: prediction={pr.get('prediction')} probability={round(pr.get('probability') or 0, 3)}")

5 of 5 scored
  3997: prediction=1 probability=0.922
  723: prediction=1 probability=0.921
  628: prediction=1 probability=0.92
  8786: prediction=1 probability=0.919
  2517: prediction=1 probability=0.919


In [7]:
charged = credits_used(ls) - before
save_metrics(".", {"notebook": "04_verified_purchase_binary", "task": "binary · review.verified", "model": model.get("model_type"),
                   "project_id": p.id, "model_id": model["model_id"], "training_duration_sec": model.get("training_duration_sec"),
                   "credits_charged_this_run": charged,
                   "headline": {"auroc": metrics.get("auroc"), "pr_auc": metrics.get("pr_auc"), "f1": metrics.get("f1"), "accuracy": metrics.get("acc")},
                   "baseline": {"auroc": 0.5, "pr_auc": metrics.get("pr_auc_baseline")}})

wrote results/metrics.json


PosixPath('results/metrics.json')